# 实践作业  
作业说明：已将关键代码贴出，可根据自身能力选择基础或进阶作业完成，想挑战自己也可两部分都完成。  


## 基础作业  
选用不同的指标，例如重构不同的损失函数、使用BULE、Gouge指标观察模型生成质量，从而控制蒸馏过程

In [2]:
#  计算 BLEU 和 ROUGE

from datasets import load_metric
import numpy as np

class KnowledgeDistillationTrainer:
    def __init__(self, *args, tokenizer=None, **kwargs):
        ...
        self.tokenizer = tokenizer
        self.bleu = load_metric("bleu")
        self.rouge = load_metric("rouge")
        self.history.update({
            'val_bleu': [],
            'val_rouge': []
        })

    def _evaluate(self, dataloader):
        self.student_model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []
        all_teacher_texts = []
        all_student_texts = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                # 教师模型生成解释
                with torch.no_grad():
                    teacher_ids = self.teacher_model.base_model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_length=50,
                        num_beams=4,
                        pad_token_id=self.tokenizer.pad_token_id
                    )
                    teacher_texts = self.tokenizer.batch_decode(teacher_ids, skip_special_tokens=True)
                    all_teacher_texts.extend(teacher_texts)

                # 学生模型生成解释
                student_ids = self.student_model.base_model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=50,
                    num_beams=4,
                    pad_token_id=self.tokenizer.pad_token_id
                )
                student_texts = self.tokenizer.batch_decode(student_ids, skip_special_tokens=True)
                all_student_texts.extend(student_texts)

                # 原始分类任务评估
                outputs = self.student_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs['loss']
                logits = outputs['logits']
                total_loss += loss.item()

                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())

        # 计算 BLEU 和 ROUGE
        bleu_scores = []
        rouge_scores = []
        for student_text, teacher_text in zip(all_student_texts, all_teacher_texts):
            # BLEU 需要 reference 是 list of list
            bleu_score = self.bleu.compute(
                predictions=[student_text],
                references=[[teacher_text]]
            )
            bleu_scores.append(bleu_score['bleu'])

            # ROUGE-L
            rouge_score = self.rouge.compute(
                predictions=[student_text],
                references=[teacher_text]
            )
            rouge_scores.append(rouge_score['rougeL'].fmeasure)

        avg_bleu = np.mean(bleu_scores)
        avg_rouge = np.mean(rouge_scores)
        accuracy = accuracy_score(all_labels, all_preds)

        # 记录指标
        self.history['val_bleu'].append(avg_bleu)
        self.history['val_rouge'].append(avg_rouge)

        return total_loss / len(dataloader), accuracy

## 进阶作业  
将科学考试任务替换为医学考试任务，评估蒸馏后的模型性能，使用挂载的数据集进行蒸馏任务

In [3]:
def process_for_training(self, tokenizer, max_length=512):
    """处理数据为训练格式，适配新数据格式"""
    if len(self.data) == 0:
        raise ValueError("没有可处理的数据，请先确保数据已正确加载")

    processed_data = {'input_ids': [], 'attention_mask': [], 'labels': []}

    logger.info(f"处理 {len(self.data)} 条数据记录...")
    for idx, item in enumerate(self.data):
        try:
            # 验证数据格式
            if 'question' not in item:
                logger.warning(f"数据项 {idx} 缺少 'question' 字段，跳过")
                continue
            if 'options' not in item or not item['options']:
                logger.warning(f"数据项 {idx} 缺少选项，跳过")
                continue
            if 'answer_idx' not in item:
                logger.warning(f"数据项 {idx} 缺少答案标记，跳过")
                continue

            question = item['question']
            options = item['options']
            answer_idx = item['answer_idx']

            # 验证答案格式是否为 A/B/C/D
            if answer_idx not in ['A', 'B', 'C', 'D']:
                logger.warning(f"数据项 {idx} 答案格式不正确: {answer_idx}，跳过")
                continue

            correct_idx = ord(answer_idx) - ord('A')  # 转换为 0~3

            # 为每个选项生成一个样本
            for i, (label, text) in enumerate(options.items()):
                # 构造输入文本
                text_input = f"问题：{question}\n选项：{text}\n这个选项是否正确？"

                # 编码文本
                encoding = tokenizer(text_input, max_length=max_length, padding='max_length', truncation=True)

                # 添加到处理后的数据中
                processed_data['input_ids'].append(encoding['input_ids'])
                processed_data['attention_mask'].append(encoding['attention_mask'])

                # 判断是否为正确选项
                is_correct = 1 if i == correct_idx else 0
                processed_data['labels'].append(is_correct)

        except Exception as e:
            logger.warning(f"处理数据项 {idx} 时出错: {str(e)}")
            continue

    # 检查是否有有效处理的数据
    if not processed_data['input_ids']:
        raise ValueError("处理后没有有效的训练数据")

    # 转换为PyTorch张量
    for key in processed_data:
        processed_data[key] = torch.tensor(processed_data[key])

    logger.info(f"数据处理完成，共生成 {len(processed_data['input_ids'])} 个训练样本")
    return processed_data